In [1]:
import io
import csv
import json
import pandas as pd
import IPython.display as display
from google.colab import files

print("=== Upload your CSV file (e.g., Student_Grade.csv) below ===")
uploaded = files.upload()

if uploaded:
    file_name = list(uploaded.keys())[0]
    file_bytes = uploaded[file_name]

    # Decode CSV text safely across common Excel encodings
    text_data = None
    for enc in ['utf-8', 'cp1252', 'latin1']:
        try:
            text_data = file_bytes.decode(enc)
            break
        except Exception:
            pass

    # Parse multi-student CSV using standard csv.reader (handles quoted strings with commas)
    reader = csv.reader(text_data.splitlines())

    students = []
    current_student = None

    for parts in reader:
        if not parts or not any(parts):
            continue

        parts = [p.strip() for p in parts]

        # Detect student header line (e.g. "Juan Dela Cruz, BS Computer Science, 1st Year")
        if len(parts) >= 3 and parts[0].lower() not in ['code', 'subject'] and parts[2].lower() not in ['units', 'unit', '']:
            if 'code' in parts[0].lower() or 'description' in parts[1].lower():
                continue

            if not any(char.isdigit() for char in parts[0]):
                current_student = {
                    'name': parts[0],
                    'course': parts[1] if len(parts) > 1 else '',
                    'year': parts[2] if len(parts) > 2 else '',
                    'subjects': []
                }
                students.append(current_student)
                continue

        # Parse subject records under active student
        if current_student is not None:
            if parts[0].lower() in ['code', 'subject']:
                continue

            non_empty = [p for p in parts if p]
            if len(non_empty) >= 5:
                # Header format: Code, Description, Units, Teacher, Grade
                c0 = non_empty[0] # Code
                c1 = non_empty[1] # Description
                c2 = non_empty[2] # Units
                c3 = non_empty[3] # Teacher
                c4 = non_empty[4] # Grade

                try:
                    units = float(c2)
                except ValueError:
                    units = 0.0

                try:
                    grade = float(c4)
                except ValueError:
                    grade = 0.0

                current_student['subjects'].append({
                    'code': c0,
                    'desc': c1,
                    'teacher': c3,
                    'units': units,
                    'grade': grade
                })

    students_json = json.dumps(students)

    raw_html = """<!DOCTYPE html>
<html>
<head>
<script src="https://cdnjs.cloudflare.com/ajax/libs/html2pdf.js/0.10.1/html2pdf.bundle.min.js"></script>

<style>
  body {
    font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, Helvetica, Arial, sans-serif;
    background-color: #f1f5f9;
    padding: 25px 0;
    margin: 0;
  }
  .gui-card {
    background: #ffffff;
    border-radius: 12px;
    box-shadow: 0 4px 25px rgba(15, 23, 42, 0.1);
    max-width: 840px;
    margin: 0 auto;
    padding: 35px 45px;
    border-top: 6px solid #2563eb;
  }
  .top-action-bar {
    display: flex;
    justify-content: space-between;
    align-items: center;
    background-color: #f0fdf4;
    border: 1px solid #bbf7d0;
    border-radius: 8px;
    padding: 10px 18px;
    margin-bottom: 20px;
  }
  .file-status {
    color: #166534;
    font-size: 14px;
    font-weight: 600;
  }
  .student-selector-bar {
    display: flex;
    align-items: center;
    gap: 12px;
    background-color: #eff6ff;
    border: 1px solid #bfdbfe;
    border-radius: 8px;
    padding: 12px 18px;
    margin-bottom: 25px;
  }
  .student-selector-bar label {
    font-size: 14px;
    font-weight: 700;
    color: #1e3a8a;
  }
  .student-selector-bar select {
    font-size: 14px;
    font-weight: 600;
    padding: 7px 12px;
    border-radius: 6px;
    border: 1px solid #93c5fd;
    outline: none;
    background-color: #ffffff;
    color: #1e293b;
    cursor: pointer;
    flex-grow: 1;
  }
  .btn-download-pdf {
    background-color: #dc2626;
    color: #ffffff;
    border: none;
    padding: 8px 16px;
    border-radius: 6px;
    font-size: 13px;
    font-weight: 600;
    cursor: pointer;
    transition: background-color 0.2s;
  }
  .btn-download-pdf:hover {
    background-color: #b91c1c;
  }
  .form-header {
    display: flex;
    justify-content: space-between;
    align-items: flex-start;
    margin-bottom: 20px;
    background-color: #f8fafc;
    padding: 15px 20px;
    border-radius: 8px;
    border: 1px solid #e2e8f0;
  }
  .form-group {
    display: flex;
    align-items: center;
    gap: 10px;
    margin-bottom: 8px;
  }
  .form-group label {
    font-size: 14px;
    font-weight: 700;
    color: #334155;
    white-space: nowrap;
  }
  .form-group input {
    border: 1px solid #cbd5e1;
    border-radius: 6px;
    background: #ffffff;
    font-size: 14px;
    font-weight: 700;
    padding: 5px 10px;
    outline: none;
    color: #0f172a !important;
  }
  .form-group input:focus {
    border-color: #2563eb;
    box-shadow: 0 0 0 2px rgba(37, 99, 235, 0.15);
  }
  .divider-line {
    height: 2px;
    background-color: #334155;
    margin: 20px 0;
  }
  table {
    width: 100%;
    border-collapse: collapse;
    margin-bottom: 20px;
  }
  th {
    text-align: left;
    font-size: 14px;
    font-weight: 800;
    color: #1e293b;
    padding: 10px 8px;
    background-color: #f1f5f9;
    border-bottom: 2px solid #cbd5e1;
  }
  td {
    padding: 6px 4px;
    vertical-align: middle;
  }
  tr:nth-child(even) td {
    background-color: #f8fafc;
  }
  td input {
    width: 95%;
    border: 1px solid #e2e8f0;
    border-radius: 4px;
    background: #ffffff;
    font-size: 13.5px;
    font-weight: 600;
    padding: 5px 7px;
    outline: none;
    color: #0f172a !important;
  }
  td input:focus {
    border-color: #2563eb;
    box-shadow: 0 0 0 2px rgba(37, 99, 235, 0.15);
  }
  .summary-section {
    display: flex;
    flex-direction: column;
    align-items: center;
    gap: 12px;
    margin-top: 20px;
    background-color: #f8fafc;
    padding: 18px;
    border-radius: 8px;
    border: 1px solid #e2e8f0;
  }
  .summary-row-top {
    display: flex;
    justify-content: space-around;
    width: 80%;
  }
  .summary-field {
    display: flex;
    align-items: center;
    gap: 8px;
    font-size: 14px;
    font-weight: 700;
    color: #334155;
  }
  .summary-field input {
    border: 1px solid #cbd5e1;
    border-radius: 6px;
    width: 75px;
    text-align: center;
    font-weight: 800;
    font-size: 15px;
    background: #ffffff;
    padding: 4px 6px;
    outline: none;
    color: #0f172a !important;
  }
  .gwa-highlight {
    background-color: #eff6ff;
    padding: 8px 16px;
    border-radius: 6px;
    border: 1px solid #bfdbfe;
  }
  .gwa-highlight label, .gwa-highlight span {
    color: #1e3a8a;
  }
  .gwa-highlight input {
    width: 90px;
    color: #2563eb !important;
    border-color: #93c5fd;
    background-color: #ffffff;
  }

  /* LEADERBOARD SECTION STYLES */
  .leaderboard-section {
    margin-top: 35px;
    background: #ffffff;
    border-radius: 10px;
    border: 1px solid #e2e8f0;
    padding: 20px;
  }
  .leaderboard-title {
    font-size: 17px;
    font-weight: 800;
    color: #1e293b;
    margin-bottom: 15px;
    display: flex;
    align-items: center;
    gap: 8px;
  }
  .leaderboard-table {
    width: 100%;
    border-collapse: collapse;
  }
  .leaderboard-table th {
    background-color: #f8fafc;
    font-size: 13px;
    font-weight: 700;
    color: #475569;
    padding: 8px 10px;
  }
  .leaderboard-table td {
    font-size: 13.5px;
    font-weight: 600;
    color: #1e293b;
    padding: 10px;
    border-bottom: 1px solid #f1f5f9;
  }
  .rank-badge {
    display: inline-block;
    width: 28px;
    text-align: center;
    font-weight: 800;
  }
  .top-1 { background-color: #fef9c3; color: #854d0e; border-radius: 50%; padding: 2px; }
  .top-2 { background-color: #f1f5f9; color: #475569; border-radius: 50%; padding: 2px; }
  .top-3 { background-color: #ffedd5; color: #9a3412; border-radius: 50%; padding: 2px; }
  .btn-view-record {
    background-color: #2563eb;
    color: white;
    border: none;
    padding: 4px 10px;
    border-radius: 4px;
    font-size: 12px;
    font-weight: 600;
    cursor: pointer;
  }
  .btn-view-record:hover {
    background-color: #1d4ed8;
  }

  @media print {
    body { background-color: #ffffff; padding: 0; }
    .gui-card { box-shadow: none; border: none; max-width: 100%; padding: 10px; }
    .top-action-bar, .student-selector-bar, .leaderboard-section { display: none !important; }
  }
</style>
</head>
<body>

<div class="gui-card" id="formArea">
  <div class="top-action-bar" data-html2canvas-ignore="true">
    <div class="file-status">📄 Loaded File: FILE_NAME_VAL (TOTAL_STUDENTS_VAL Students Found)</div>
    <button class="btn-download-pdf" id="pdfBtn" onclick="downloadPDF()">📄 Download Current Student PDF</button>
  </div>

  <div class="student-selector-bar" data-html2canvas-ignore="true">
    <label for="studentSelect">Select Student Record:</label>
    <select id="studentSelect" onchange="loadStudentRecord(this.value)"></select>
  </div>

  <div class="form-header">
    <div class="form-group">
      <label>Student name:</label>
      <input type="text" id="studentName" style="width: 220px;">
    </div>
    <div style="display: flex; flex-direction: column; align-items: flex-end;">
      <div class="form-group">
        <label>Course:</label>
        <input type="text" id="studentCourse" style="width: 200px;">
      </div>
      <div class="form-group">
        <label>Year:</label>
        <input type="text" id="studentYear" style="width: 120px;">
      </div>
    </div>
  </div>

  <div class="divider-line"></div>

  <table>
    <thead>
      <tr>
        <th style="width: 5%;"></th>
        <th style="width: 18%;">Subject</th>
        <th style="width: 37%;">Description</th>
        <th style="width: 22%;">Teacher</th>
        <th style="width: 9%;">Units</th>
        <th style="width: 9%;">Grade</th>
      </tr>
    </thead>
    <tbody id="tableBody"></tbody>
  </table>

  <div class="divider-line"></div>

  <div class="summary-section">
    <div class="summary-row-top">
      <div class="summary-field">
        <span>Total Units:</span>
        <input type="text" id="totalUnits" readonly>
      </div>
      <div class="summary-field">
        <span>#of subject:</span>
        <input type="text" id="totalSubjects" readonly>
      </div>
    </div>
    <div class="summary-field gwa-highlight">
      <span>Grade Weighted Average:</span>
      <input type="text" id="gwaValue" readonly>
    </div>
  </div>

  <!-- LEADERBOARD SECTION -->
  <div class="leaderboard-section" data-html2canvas-ignore="true">
    <div class="leaderboard-title">🏆 Student Academic Standing Leaderboard (Highest to Lowest)</div>
    <table class="leaderboard-table">
      <thead>
        <tr>
          <th style="width: 10%; text-align: center;">Rank</th>
          <th style="width: 35%;">Student Name</th>
          <th style="width: 30%;">Course & Year</th>
          <th style="width: 13%; text-align: center;">GWA</th>
          <th style="width: 12%; text-align: center;">Action</th>
        </tr>
      </thead>
      <tbody id="leaderboardBody"></tbody>
    </table>
  </div>
</div>

<script>
  let allStudents = STUDENTS_JSON_HERE;
  let activeStudentIndex = 0;

  function initDropdown() {
    const selector = document.getElementById('studentSelect');
    selector.innerHTML = '';

    allStudents.forEach((st, idx) => {
      const opt = document.createElement('option');
      opt.value = idx;
      opt.textContent = `${idx + 1}. ${st.name} — ${st.course} (${st.year})`;
      selector.appendChild(opt);
    });

    if (allStudents.length > 0) {
      loadStudentRecord(0);
    }
  }

  function loadStudentRecord(index) {
    activeStudentIndex = parseInt(index);
    document.getElementById('studentSelect').value = activeStudentIndex;

    const student = allStudents[activeStudentIndex];
    if (!student) return;

    document.getElementById('studentName').value = student.name || '';
    document.getElementById('studentCourse').value = student.course || '';
    document.getElementById('studentYear').value = student.year || '';

    const tbody = document.getElementById('tableBody');
    tbody.innerHTML = '';

    student.subjects.forEach((sub, idx) => {
      const tr = document.createElement('tr');
      tr.innerHTML = `
        <td style="font-weight: 700; color: #475569; text-align: center;">${idx + 1}.</td>
        <td><input type="text" class="inp-code" value="${sub.code}"></td>
        <td><input type="text" class="inp-desc" value="${sub.desc}"></td>
        <td><input type="text" class="inp-teach" value="${sub.teacher}"></td>
        <td><input type="text" class="inp-units" value="${sub.units}" oninput="calculate()"></td>
        <td><input type="text" class="inp-grade" value="${sub.grade}" oninput="calculate()"></td>
      `;
      tbody.appendChild(tr);
    });

    calculate();
  }

  function calculate() {
    const unitInputs = document.querySelectorAll('.inp-units');
    const gradeInputs = document.querySelectorAll('.inp-grade');

    let totalUnits = 0;
    let weightedSum = 0;
    let subjectCount = 0;

    const currentStudent = allStudents[activeStudentIndex];
    if (currentStudent) {
      currentStudent.name = document.getElementById('studentName').value;
      currentStudent.course = document.getElementById('studentCourse').value;
      currentStudent.year = document.getElementById('studentYear').value;
      currentStudent.subjects = [];
    }

    const codes = document.querySelectorAll('.inp-code');
    const descs = document.querySelectorAll('.inp-desc');
    const teaches = document.querySelectorAll('.inp-teach');

    for (let i = 0; i < unitInputs.length; i++) {
      const u = parseFloat(unitInputs[i].value);
      const g = parseFloat(gradeInputs[i].value);

      if (currentStudent) {
        currentStudent.subjects.push({
          code: codes[i].value,
          desc: descs[i].value,
          teacher: teaches[i].value,
          units: !isNaN(u) ? u : 0,
          grade: !isNaN(g) ? g : 0
        });
      }

      if (!isNaN(u) && !isNaN(g)) {
        totalUnits += u;
        weightedSum += (u * g);
        subjectCount++;
      }
    }

    const calculatedGWA = totalUnits > 0 ? (weightedSum / totalUnits).toFixed(2) : '0.00';

    document.getElementById('totalUnits').value = totalUnits ? totalUnits.toFixed(1) : '';
    document.getElementById('totalSubjects').value = subjectCount ? subjectCount : '';
    document.getElementById('gwaValue').value = (totalUnits > 0) ? calculatedGWA : '';

    updateLeaderboard();
  }

  function updateLeaderboard() {
    const lbBody = document.getElementById('leaderboardBody');
    lbBody.innerHTML = '';

    // Calculate GWA for all students
    let rankedList = allStudents.map((st, originalIndex) => {
      let totU = 0;
      let wSum = 0;
      st.subjects.forEach(sub => {
        const u = parseFloat(sub.units);
        const g = parseFloat(sub.grade);
        if (!isNaN(u) && !isNaN(g)) {
          totU += u;
          wSum += (u * g);
        }
      });
      const gwaVal = totU > 0 ? (wSum / totU) : 99.0;
      return {
        originalIndex: originalIndex,
        name: st.name || 'Unnamed',
        course: st.course || '',
        year: st.year || '',
        gwa: gwaVal
      };
    });

    // Lower GWA number is better standing (1.00 is top mark)
    rankedList.sort((a, b) => a.gwa - b.gwa);

    rankedList.forEach((st, rankIdx) => {
      const tr = document.createElement('tr');
      let rankDisplay = `${rankIdx + 1}`;
      let badgeClass = '';

      if (rankIdx === 0) { rankDisplay = '🥇 1'; badgeClass = 'top-1'; }
      else if (rankIdx === 1) { rankDisplay = '🥈 2'; badgeClass = 'top-2'; }
      else if (rankIdx === 2) { rankDisplay = '🥉 3'; badgeClass = 'top-3'; }

      tr.innerHTML = `
        <td style="text-align: center;"><span class="rank-badge ${badgeClass}">${rankDisplay}</span></td>
        <td style="font-weight: 700;">${st.name}</td>
        <td style="color: #475569;">${st.course} (${st.year})</td>
        <td style="text-align: center; font-weight: 800; color: #2563eb;">${st.gwa < 90 ? st.gwa.toFixed(2) : 'N/A'}</td>
        <td style="text-align: center;">
          <button class="btn-view-record" onclick="loadStudentRecord(${st.originalIndex})">View</button>
        </td>
      `;
      lbBody.appendChild(tr);
    });
  }

  function downloadPDF() {
    const btn = document.getElementById('pdfBtn');
    btn.textContent = '⏳ Generating PDF...';

    const sName = document.getElementById('studentName').value.trim() || 'Student';
    const filename = `${sName.replace(/[^a-zA-Z0-9]/g, '_')}_Grade_Report.pdf`;

    const element = document.getElementById('formArea');
    const opt = {
      margin:       [0.3, 0.3, 0.3, 0.3],
      filename:     filename,
      image:        { type: 'jpeg', quality: 0.98 },
      html2canvas:  { scale: 2, logging: false },
      jsPDF:        { unit: 'in', format: 'letter', orientation: 'portrait' }
    };

    html2pdf().set(opt).from(element).save().then(() => {
      btn.textContent = '📄 Download Current Student PDF';
    });
  }

  initDropdown();
</script>

</body>
</html>
"""

    final_html = raw_html.replace('FILE_NAME_VAL', str(file_name))
    final_html = final_html.replace('TOTAL_STUDENTS_VAL', str(len(students)))
    final_html = final_html.replace('STUDENTS_JSON_HERE', students_json)

    display.display(display.HTML(final_html))

else:
   print("No file was uploaded.")


=== Upload your CSV file (e.g., Student_Grade.csv) below ===


Saving Tando_10_Students.CSV.csv to Tando_10_Students.CSV.csv
